# R4 - Short-term + long-term memory for the claims agent + evidence

GenAi-2 (Paper 5) R4: give the R2/R3 claims agent memory, run over the client's memory probes (user_id `adjuster.demo@saffronshield.example`).

Deliverables:
1. `evidence/r4_memory.json` - keys `backend` (lakebase, or delta on the permitted fallback), `user_id` (from memory_probes.json) and `sessions` (two objects, session 1 first). Each session: `session`, `thread_id` (memory-session-1 / memory-session-2), `turns`; each turn: `turn` (from 1 within the session), `question`, `answer`, `tool_calls` (list of tool names), `thread_messages_before` (messages the conversation held before the turn, 0 on a first turn).
2. Memory checks - session 1 turn 3 names the claim (CLM-10556, short-term, answered from carried history); session 2 turn 1 calls recall_notes on an empty thread (0 messages before) and answers with ADJ-2219 from the notes store (long-term). The tools are named save_note and recall_notes.
3. `claims.genai.adjuster_notes` - delta fallback table with a note (STRING) column holding the saved notes.

How it works: short-term memory = the growing message array of one thread, dropped when the session ends; long-term memory = notes saved via save_note into the delta table, read back via recall_notes in any later session.

In [0]:
# notebook-scoped install (same pattern as R2/R3; known to downgrade mlflow - accepted)
%pip install langchain-core langchain-databricks

In [0]:
# R4 config (same constants as R1-R3) + the memory probes, evidence and notes-table paths.
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
TEMPERATURE = 0.0            # memory runs must be inspectable and repeatable
BACKEND = "delta"            # lakebase preferred by the client; delta is the permitted fallback

GENAI2_ROOT = "/Workspace/Users/custom001x@gmail.com/saffron-shield/GenAi-2"
PROMPT_V1_PATH = GENAI2_ROOT + "/agent/prompts/system_v1.md"   # R2's prompt (unchanged)
PROMPT_V2_PATH = GENAI2_ROOT + "/agent/prompts/system_v2.md"   # v1 + the memory rule
PROBES_PATH = "/Volumes/claims/raw/landing/genai/memory_probes.json"
EVIDENCE_PATH = GENAI2_ROOT + "/evidence/r4_memory.json"
NOTES_TABLE = "claims.genai.adjuster_notes"

import mlflow
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Users/custom001x@gmail.com/saffron-shield/genai2-agent")
EXP = mlflow.get_experiment_by_name("/Users/custom001x@gmail.com/saffron-shield/genai2-agent")
print(f"CATALOG={CATALOG} | CHAT_MODEL={CHAT_MODEL} | temperature={TEMPERATURE} | backend={BACKEND}")
print(f"MLflow experiment id={EXP.experiment_id}")

In [0]:
# 2. system_v2.md = system_v1.md + one memory rule; the notes table (reset each run);
#    and the client probes with user_id, sessions and checks.
import json

V1 = dbutils.fs.head(PROMPT_V1_PATH, 100000)
MEMORY_RULE = ("4. The user asks to save, note or remember something - call save_note with the "
               "note text (include the claim id when the note is about a claim) and confirm it "
               "was saved. The user asks about a note, a reference, or anything they were told "
               "or mentioned in an earlier conversation - ALWAYS call recall_notes FIRST and "
               "answer only from the notes it returns; saved notes and references are NOT in "
               "lookup_claim or any claim record. Never claim a note was saved without calling "
               "save_note, and never invent note contents.")
assert "4." not in V1, "v1 already has a rule 4 - refusing to stack a second one"
dbutils.fs.put(PROMPT_V2_PATH, V1.rstrip() + "\n" + MEMORY_RULE + "\n", overwrite=True)
print(f"{PROMPT_V2_PATH} written (v1 + memory rule)")

probes_doc = json.loads(dbutils.fs.head(PROBES_PATH, 100000))
USER_ID = probes_doc["user_id"]
print(f"user_id={USER_ID}")
for sd in probes_doc["sessions"]:
    print(f"  session {sd['session']} ({sd['thread_id']}): {len(sd['turns'])} turn(s)")
for chk in probes_doc["checks"]:
    print(f"  check: S{chk['session']}T{chk['turn']} {chk['kind']} must_contain={chk['must_contain']}"
          + (f" tool={chk['tool']}" if chk.get("tool") else ""))

spark.sql(f"""
CREATE OR REPLACE TABLE {NOTES_TABLE} (
  note_id  STRING    COMMENT 'unique note id',
  user_id  STRING    COMMENT 'adjuster who saved the note',
  claim_id STRING    COMMENT 'claim the note is about, NULL when general',
  note     STRING    COMMENT 'the saved note text',
  saved_at TIMESTAMP COMMENT 'when the note was saved'
)
COMMENT 'R4 long-term memory: notes saved by adjusters via the save_note tool (delta fallback backend)'
""")
print(f"{NOTES_TABLE} created (reset for a reproducible run)")

In [0]:
# 3. Six tools: the four R1 UC-function tools + R4's save_note / recall_notes over the delta
#    notes table. run_turn carries the thread's message history (short-term memory).
try:
    mlflow.langchain.autolog()   # captures the LLM calls as spans inside our traces
except Exception as e:
    print(f"langchain autolog unavailable ({e}); TOOL/CHAIN spans still trace every run")

import json
import re
import uuid

from mlflow.entities import SpanType
from langchain_databricks import ChatDatabricks
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

SYSTEM_PROMPT = dbutils.fs.head(PROMPT_V2_PATH, 100000)

def q(s) -> str:
    """SQL-escape a string literal."""
    return str(s).replace("'", "''")

@tool
def lookup_claim(claim_id: str) -> str:
    "Look up ONE insurance claim by its exact claim id (example: CLM-10642). Returns every stored fact of that claim as JSON: member_id, policy_id, hospital_id, status, claim_type, admission and discharge dates, billed and approved amounts, plan code at admission, room rent and cap, network flag, policy start date and tenure. Use for ANY question about a specific claim."
    df = spark.sql(f"SELECT * FROM claims.genai.lookup_claim('{q(claim_id)}')")
    return json.dumps([r.asDict() for r in df.collect()], default=str)

@tool
def plan_at(member_id: str, as_of_date: str) -> str:
    "Look up the Saffron Shield plan terms that were VALID on a given date (YYYY-MM-DD) for a member id (example: MEM-0493). Returns plan_code, plan_family, room_rent_cap_per_day, copay_pct, ped_waiting_months, initial_waiting_days, valid_from, valid_to. Use with the member id and admission date from lookup_claim when a claim question needs plan terms."
    df = spark.sql(f"SELECT * FROM claims.genai.plan_at('{q(member_id)}', DATE'{q(as_of_date)}')")
    return json.dumps([r.asDict() for r in df.collect()], default=str)

@tool
def search_policy(query: str) -> str:
    "Search the Saffron Shield policy binder (hybrid keyword + semantic). Returns up to 5 chunks with chunk_id, doc_name, heading and full text. Use for ANY question about what the policy says: waiting periods, room rent caps, co-payments, network hospitals, exclusions. Always read each chunk's 'Applies to' line before quoting it."
    df = spark.sql(f"SELECT * FROM claims.genai.search_policy('{q(query)}')")
    return json.dumps([r.asDict() for r in df.collect()], default=str)

@tool
def compute_payable(plan_code: str, is_pre_existing: bool, policy_tenure_months: int,
                    days_since_policy_start: int, is_network_at_admission: bool,
                    room_rent_per_day: int, billed_amount: float) -> str:
    "Decide how much a claim pays, strictly by the policy Definitions. Pass the claim facts from lookup_claim (tenure months, days since policy start, pre-existing flag, network flag, room rent per day, billed amount) and the plan_code from plan_at. Returns JSON with payable, reason, room_rent_deduction_pct, copay_pct and payable_amount. Never decide a payout without this tool."
    pe = "true" if is_pre_existing else "false"
    nw = "true" if is_network_at_admission else "false"
    row = spark.sql(f"""
      SELECT claims.genai.compute_payable('{q(plan_code)}', {pe}, {int(policy_tenure_months)},
             {int(days_since_policy_start)}, {nw}, {int(room_rent_per_day)},
             CAST({float(billed_amount)} AS DOUBLE)) AS s
    """).first()
    return row.s

@tool
def save_note(note: str) -> str:
    "Save a note for the current adjuster so it is remembered in FUTURE sessions, even after this conversation ends. Pass the full note text; include the claim id when the note is about a claim. Use whenever the user asks to save, note, remember or jot down something."
    tokens = re.split("[^A-Za-z0-9-]+", note)
    claim_id = next((t for t in tokens if t.startswith("CLM-")), None)
    claim_sql = f"'{q(claim_id)}'" if claim_id else "NULL"
    note_id = "NT-" + uuid.uuid4().hex[:10]
    spark.sql(f"INSERT INTO {NOTES_TABLE} VALUES ('{q(note_id)}', '{q(USER_ID)}', {claim_sql}, '{q(note)}', current_timestamp())")
    return json.dumps({"saved": True, "note_id": note_id, "claim_id": claim_id})

@tool
def recall_notes() -> str:
    "Recall every note the current adjuster has saved in past sessions. Returns all saved notes with their note id, claim id, text and save time. Use whenever the user asks about something they noted or saved earlier."
    df = spark.sql(f"SELECT note_id, claim_id, note, saved_at FROM {NOTES_TABLE} WHERE user_id = '{q(USER_ID)}' ORDER BY saved_at")
    return json.dumps([r.asDict() for r in df.collect()], default=str)

TOOL_FNS = {t.name: t for t in [lookup_claim, plan_at, search_policy, compute_payable,
                                save_note, recall_notes]}
assert set(TOOL_FNS) == {"lookup_claim", "plan_at", "search_policy", "compute_payable",
                         "save_note", "recall_notes"}

CALLS = []   # every tool invocation of the current turn: [{name, args}]

@mlflow.trace(span_type=SpanType.CHAIN)
def run_turn(history: list, question: str) -> dict:
    """One traced agent turn on a thread: history + question -> tool loop -> answer.
    history is the thread's short-term memory (every message so far, system message excluded)."""
    CALLS.clear()
    llm = ChatDatabricks(endpoint=CHAT_MODEL, temperature=TEMPERATURE).bind_tools(list(TOOL_FNS.values()))
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + list(history)
    n_before = len(history)          # recorded as thread_messages_before
    messages.append(HumanMessage(content=question))
    final_answer = None
    for _step in range(6):
        ai = llm.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            final_answer = ai.content
            break
        for tc in ai.tool_calls:
            CALLS.append({"name": tc["name"], "args": tc["args"]})
            out = TOOL_FNS[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))
    assert final_answer is not None, "agent never produced a final answer"
    return {"answer": final_answer, "tool_calls": list(CALLS),
            "thread_messages_before": n_before, "new_messages": messages[1:]}

print("agent ready: tools =", sorted(TOOL_FNS))

In [0]:
# 4. Run both sessions: history grows inside a thread (short-term memory) and is dropped
#    between sessions; notes live in the delta table (long-term memory).
def run_session(sd: dict) -> dict:
    history = []            # this thread's short-term memory; gone when the session ends
    turns = []
    for i, question in enumerate(sd["turns"], start=1):
        r = run_turn(history, question)
        history.extend(r["new_messages"])
        turns.append({"turn": i, "question": question, "answer": r["answer"],
                      "tool_calls": [c["name"] for c in r["tool_calls"]],
                      "thread_messages_before": r["thread_messages_before"]})
        print(f"  S{sd['session']} T{i} | messages before = {r['thread_messages_before']} | tools = {[c['name'] for c in r['tool_calls']]}")
        print(f"     answer: {r['answer'][:100]}")
    return {"session": sd["session"], "thread_id": sd["thread_id"], "turns": turns}

sessions_out = []
for sd in probes_doc["sessions"]:   # probes file already lists session 1 first
    print(f"session {sd['session']} ({sd['thread_id']})")
    sessions_out.append(run_session(sd))

n_notes = spark.sql(f"SELECT COUNT(*) AS c FROM {NOTES_TABLE}").first().c
print(f"\n{len(sessions_out)} sessions run | notes in table: {n_notes}")

In [0]:
# 5. Write evidence/r4_memory.json, then verify every R4 rule from the PERSISTED file.
import json

ev = {"backend": BACKEND, "user_id": USER_ID, "sessions": sessions_out}
dbutils.fs.put(EVIDENCE_PATH, json.dumps(ev, indent=2), overwrite=True)

loaded = json.loads(dbutils.fs.head(EVIDENCE_PATH, 500000))
assert list(loaded.keys()) == ["backend", "user_id", "sessions"], list(loaded.keys())
assert loaded["backend"] in ("lakebase", "delta"), loaded["backend"]
assert loaded["user_id"] == probes_doc["user_id"], loaded["user_id"]

assert [s["session"] for s in loaded["sessions"]] == [1, 2], "two sessions, session 1 first"
s1, s2 = loaded["sessions"]
for s in (s1, s2):
    assert list(s.keys()) == ["session", "thread_id", "turns"], list(s.keys())
assert s1["thread_id"] == "memory-session-1" and s2["thread_id"] == "memory-session-2"
assert s1["thread_id"] != s2["thread_id"], "the two thread_ids must differ"

for s in loaded["sessions"]:
    for j, t in enumerate(s["turns"], start=1):
        assert list(t.keys()) == ["turn", "question", "answer", "tool_calls", "thread_messages_before"], list(t.keys())
        assert t["turn"] == j, t["turn"]
        assert t["question"].strip() and t["answer"].strip(), t
        assert isinstance(t["tool_calls"], list), t

# the two client checks, exactly as listed in memory_probes.json
for chk in probes_doc["checks"]:
    s = loaded["sessions"][chk["session"] - 1]
    t = s["turns"][chk["turn"] - 1]
    assert chk["must_contain"] in t["answer"], \
        f"{chk['kind']} check: '{chk['must_contain']}' not in answer: {t['answer'][:120]}"
    if chk["kind"] == "short_term":
        assert t["thread_messages_before"] > 0, "short-term answer must come from carried history"
        print(f"short-term check S{chk['session']}T{chk['turn']}: '{chk['must_contain']}' in answer "
              f"| messages before = {t['thread_messages_before']}")
    else:
        assert t["thread_messages_before"] == 0, "long-term answer must come from the notes store, not history"
        assert chk["tool"] in t["tool_calls"], f"expected {chk['tool']} in tool_calls: {t['tool_calls']}"
        print(f"long-term check S{chk['session']}T{chk['turn']}: '{chk['must_contain']}' in answer "
              f"| {chk['tool']} called | messages before = 0")

rows = spark.sql(f"SELECT note FROM {NOTES_TABLE}").collect()
assert any("ADJ-2219" in r.note for r in rows), "saved note must be in the delta table"
print(f"notes table holds the saved note ({len(rows)} row(s))")
print(f"evidence written: {EVIDENCE_PATH}")
print("ALL R4 SELF-CHECKS PASSED")